# S26 · Good vs bad charts

We put a misleading chart next to an honest remake of the *same* data, so you can
see that the trick is in the chart, not the numbers. Then we draw a fan chart that
shows uncertainty honestly instead of pretending we know the future exactly. Just
`matplotlib`.

**New here? Read this once.**

- New to Python? You can run this whole notebook. Press play on each cell, top to
  bottom, and read the plain-English note above each one. The point is what the
  pictures do, not the code.
- No new maths today. If you have met the idea of an honest chart before, this will
  feel familiar; if not, `primers/choosing_a_chart.md` gives the ten-minute
  picture-first version.
- Already confident with code? Look for the cell marked **Stretch (optional)** near
  the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook only uses matplotlib and numpy, which Colab already ships,
# so there is nothing to install here.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                       # fast maths on lists of numbers
import matplotlib.pyplot as plt           # drawing every chart in this notebook

## Step 1 — the data (it never changes)

Four quarters of sales, in rupees lakh. We will show this *one* dataset in an
honest way and a misleading way, so the point is unmistakable: the trick is in the
chart, not in the numbers.

In [ ]:
# Set a seed so the random series later is the same every run.
np.random.seed(0)

quarters = ["Q1", "Q2", "Q3", "Q4"]
sales    = [101.0, 102.0, 103.0, 104.5]    # sales in Rs lakh

# The real change from Q1 to Q4, as a percentage.
percent_change = (sales[3] - sales[0]) / sales[0] * 100
print("Sales (Rs lakh):", sales)
print("Real change Q1 to Q4:", round(percent_change, 1), "%")

## Step 2 — the misleading chart (truncated y-axis)

The oldest trick in the book: start the value axis high (here at 100) instead of 0.
A tiny 3.5% rise then looks like an explosion. The numbers are perfectly honest; the
picture is not.

In [ ]:
plt.figure(figsize=(6, 5))
plt.bar(quarters, sales, color="#C0392B", width=0.6)

# The dishonest part: the axis starts at 100, not 0.
plt.ylim(100, 105)

plt.ylabel("sales (Rs lakh)")
plt.title("Misleading: y-axis starts at 100  ->  'sales exploded!'")
plt.show()

## Step 3 — the honest remake (same data, axis from 0)

Exactly the same four numbers. Starting the axis at 0 shows the truth: a small,
steady rise. Bar charts should almost always start at zero.

In [ ]:
plt.figure(figsize=(6, 5))
plt.bar(quarters, sales, color="#2E9E5B", width=0.6)

# The honest fix: start the axis at 0.
plt.ylim(0, 120)

plt.ylabel("sales (Rs lakh)")
plt.title("Honest: y-axis starts at 0  ->  'sales up ~3.5%'")
plt.show()

## Step 4 — side by side, so the difference is obvious

`plt.subplots` puts two charts in one figure. Same data on the left and the right;
only the y-axis range differs. This is the picture to remember and to watch for in
other people's slides.

In [ ]:
# 1 row, 2 columns of charts. Only the y-axis range will differ.
fig, (left_chart, right_chart) = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: the misleading version.
left_chart.bar(quarters, sales, color="#C0392B", width=0.6)
left_chart.set_ylim(100, 105)
left_chart.set_title("Misleading (axis at 100)")
left_chart.set_ylabel("sales (Rs lakh)")

# Right: the honest version.
right_chart.bar(quarters, sales, color="#2E9E5B", width=0.6)
right_chart.set_ylim(0, 120)
right_chart.set_title("Honest (axis at 0)")

fig.suptitle("Same four numbers, two very different stories")
plt.show()

## Step 5 — a second trick: cherry-picking the window

We make a longer series of monthly figures that is basically flat, with one lucky
bump in the middle. Showing only the bump tells a triumphant story; showing the
whole series tells the truth.

In [ ]:
# Three years of monthly numbers, mostly flat random wandering.
months = np.arange(0, 36)
series = 100 + np.cumsum(np.random.normal(0.05, 1.4, size=36))

# Add a flattering bump in months 18..23.
series[18:24] = series[18:24] + np.linspace(0, 7, 6)

print("We have", len(series), "months of data.")
print("Months 18-23 contain a lucky upward bump.")

## Step 6 — cherry-picked window vs the full series

Left: only the 6 flattering months, which looks like a rocket. Right: all 36
months, with that same window shaded, where the bump is just a blip. Always show
enough context for the trend to be judged fairly.

In [ ]:
fig, (cherry, full) = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: only the flattering 6-month window.
cherry.plot(months[18:24], series[18:24], color="#C0392B", linewidth=3)
cherry.set_title("Cherry-picked 6 months")
cherry.set_xlabel("month")
cherry.set_ylabel("value")

# Right: the whole series, with the cherry-picked window shaded for comparison.
full.plot(months, series, color="#2E75B6", linewidth=2)
full.axvspan(18, 23, color="#F2C14E", alpha=0.3)   # shade the window
full.set_title("Full 3-year series (context restored)")
full.set_xlabel("month")

fig.suptitle("Cherry-picked windows hide the real trend")
plt.show()

## Step 7 — communicating uncertainty: a point vs a fan

A forecast is never a single certain number. We have actual revenue for months 0 to
4, and we forecast months 4 to 6. First we set up the actuals, the forecast's
most-likely path, and how wide the uncertainty is at each future month.

In [ ]:
# Actual revenue we have already seen, months 0..4 (Rs lakh).
actual_months = np.array([0, 1, 2, 3, 4])
actual_values = np.array([100.0, 103.0, 107.0, 110.0, 114.0])

# Forecast for months 4..6 (start at the last actual so the lines join up).
forecast_months = np.array([4, 5, 6])
most_likely     = np.array([114.0, 118.0, 122.0])   # the central guess
half_range      = np.array([0.0, 4.0, 9.0])          # uncertainty grows with time

print("Most-likely forecast for month 6:", most_likely[-1], "Rs lakh")
print("But the range there is", most_likely[-1] - 2 * half_range[-1],
      "to", most_likely[-1] + 2 * half_range[-1], "Rs lakh")

## Step 8 — the bare point estimate (false certainty)

A single dashed line saying "it will be 122" looks confident and is almost certainly
wrong to that precision. This is what NOT to present.

In [ ]:
plt.figure(figsize=(6.5, 5))

# The actuals.
plt.plot(actual_months, actual_values, "-o", color="#2E75B6", linewidth=2, label="actual")

# The forecast as one confident line.
plt.plot(forecast_months, most_likely, "--o", color="#C0392B", linewidth=2, label="forecast")

plt.annotate("'It will be 122.'", xy=(6, 122), xytext=(3.0, 128),
             color="#C0392B",
             arrowprops=dict(arrowstyle="->", color="#C0392B"))

plt.xlabel("month")
plt.ylabel("revenue (Rs lakh)")
plt.title("Bare point estimate: false certainty")
plt.ylim(95, 140)
plt.legend()
plt.show()

## Step 9 — the fan chart (honest about uncertainty)

`plt.fill_between` shades a band between a low line and a high line. We draw a wide
band (the full range of outcomes) and a narrower band (the most-likely zone), so the
reader sees the uncertainty grow into the future. Instead of "it will be 122" we can
now say "an 8 to 12% lift, most likely around 10%".

In [ ]:
plt.figure(figsize=(6.5, 5))

# The actuals, same as before.
plt.plot(actual_months, actual_values, "-o", color="#2E75B6", linewidth=2, label="actual")

# Wide band: most_likely plus/minus 2 * half_range (the full range of outcomes).
plt.fill_between(forecast_months,
                 most_likely - 2 * half_range,
                 most_likely + 2 * half_range,
                 color="#9DC3E6", alpha=0.5, label="range of outcomes")

# Narrow band: most_likely plus/minus 1 * half_range (the most-likely zone).
plt.fill_between(forecast_months,
                 most_likely - half_range,
                 most_likely + half_range,
                 color="#2E75B6", alpha=0.4, label="most likely")

# The central path as a dashed line through the middle.
plt.plot(forecast_months, most_likely, "--", color="#1F4E79", linewidth=2)

plt.annotate("'8-12% lift,\nmost likely ~10%'", xy=(6, 122), xytext=(2.7, 132),
             color="#1F4E79",
             arrowprops=dict(arrowstyle="->", color="#1F4E79"))

plt.xlabel("month")
plt.ylabel("revenue (Rs lakh)")
plt.title("Fan chart: honest about uncertainty")
plt.ylim(95, 140)
plt.legend(loc="lower right")
plt.show()

### Stretch (optional) — where the widening band comes from

Skip this if you just want to present well. The band widens each month because
uncertainty compounds: the further out you forecast, the less sure you are. The wide
band here is the central guess plus or minus two "half-ranges". If you think of the
spread as a bell-shaped distribution, plus or minus two standard deviations covers
about 95% of likely outcomes, so the wide band is roughly a "95% range". Below we
just print how wide that full range is at each future month, to make the widening
concrete.

In [ ]:
# Full width of the wide band = 2 * half_range on each side = 4 * half_range total.
for month, spread in zip(forecast_months, half_range):
    full_width = 4 * spread
    print("month", month, ": full range width =", round(full_width, 1), "Rs lakh")

## What you just did

You saw how the *same* numbers can mislead (a truncated axis, a cherry-picked
window) and how to remake them honestly. You also turned a single forecast number
into a fan chart that tells the truth about uncertainty. Honest charts are not just
the ethical choice; they protect you when reality lands a little off your central
guess. Take these habits into the lab when you redraw a chart from one of your own
projects.